# Week 3: Forward Selection, Backward Selection, PCR, and PLSR

In this notebook, I apply several regression-based feature selection and dimensionality reduction methods to the diabetes and Alzheimer's datasets. The goal is to compare traditional linear regression with forward selection, backward selection, Principal Component Regression (PCR), and Partial Least Squares Regression (PLSR).

The main evaluation metrics used are R² and RMSE. R² measures how much variation in the target variable is explained by the model, while RMSE measures the average prediction error in the same units as the target variable.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt

## Diabetes Dataset

The first dataset used in this notebook is the diabetes health indicators dataset. This dataset includes health, lifestyle, and demographic variables that may be useful for predicting diabetes status.

In [ ]:
diabetes_df = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")

diabetes_df.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [ ]:
print("Diabetes dataset shape:", diabetes_df.shape)
print("\nColumn names:")
print(diabetes_df.columns)

print("\nMissing values:")
print(diabetes_df.isnull().sum())

Diabetes dataset shape: (253680, 22)

Column names:
Index(['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='str')

Missing values:
Diabetes_012            0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64


## Diabetes Modeling Setup

The target variable is `Diabetes_012`, which represents diabetes status. The remaining columns are used as predictors. I use an 80/20 train-test split so the models can be evaluated on data that was not used during training.

In [ ]:
X_diabetes = diabetes_df.drop("Diabetes_012", axis=1)
y_diabetes = diabetes_df["Diabetes_012"]

# Train-test split
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diabetes,
    y_diabetes,
    test_size=0.2,
    random_state=42
)

print("Training feature shape:", X_train_d.shape)
print("Testing feature shape:", X_test_d.shape)
print("Training target shape:", y_train_d.shape)
print("Testing target shape:", y_test_d.shape)

Training feature shape: (202944, 21)
Testing feature shape: (50736, 21)
Training target shape: (202944,)
Testing target shape: (50736,)


## Baseline Linear Regression: Diabetes Dataset

Before applying feature selection or dimensionality reduction, I first fit a standard linear regression model using all available predictors. This gives a baseline R² and RMSE for comparison.

In [ ]:
# Baseline linear regression using all diabetes predictors

baseline_lr_d = LinearRegression()
baseline_lr_d.fit(X_train_d, y_train_d)

# Make predictions
y_pred_baseline_d = baseline_lr_d.predict(X_test_d)

# Evaluate model
baseline_r2_d = r2_score(y_test_d, y_pred_baseline_d)
baseline_rmse_d = np.sqrt(mean_squared_error(y_test_d, y_pred_baseline_d))

print("Diabetes Baseline Linear Regression R²:", baseline_r2_d)
print("Diabetes Baseline Linear Regression RMSE:", baseline_rmse_d)

Diabetes Baseline Linear Regression R²: 0.17330978646059458
Diabetes Baseline Linear Regression RMSE: 0.6322608048598488


## Forward Selection: Diabetes Dataset

Forward selection begins with no predictors and adds one feature at a time. At each step, the feature that produces the lowest validation RMSE is added to the model. This process continues until adding another feature no longer improves validation performance.

In [ ]:
# Create a smaller training/validation split for feature selection

X_train_fs_d, X_val_fs_d, y_train_fs_d, y_val_fs_d = train_test_split(
    X_train_d,
    y_train_d,
    test_size=0.2,
    random_state=42
)

print("Forward selection training shape:", X_train_fs_d.shape)
print("Forward selection validation shape:", X_val_fs_d.shape)

Forward selection training shape: (162355, 21)
Forward selection validation shape: (40589, 21)


In [45]:
# Forward selection function using validation RMSE

def forward_selection(X_train, y_train, X_val, y_val):
    remaining_features = list(X_train.columns)
    selected_features = []
    best_rmse = float("inf")
    history = []

    while remaining_features:
        feature_results = []

        for feature in remaining_features:
            current_features = selected_features + [feature]

            model = LinearRegression()
            model.fit(X_train[current_features], y_train)

            val_pred = model.predict(X_val[current_features])
            val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

            feature_results.append((feature, val_rmse))

        # Choose the feature with the lowest validation RMSE
        best_feature, current_best_rmse = min(feature_results, key=lambda x: x[1])

        # Stop if validation RMSE does not improve
        if current_best_rmse < best_rmse:
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            best_rmse = current_best_rmse

            history.append({
                "Step": len(selected_features),
                "Added Feature": best_feature,
                "Validation RMSE": best_rmse,
                "Selected Features": selected_features.copy()
            })

            print(f"Step {len(selected_features)}: Added {best_feature}, Validation RMSE = {best_rmse}")
        else:
            break

    return selected_features, pd.DataFrame(history)


forward_features_d, forward_history_d = forward_selection(
    X_train_fs_d,
    y_train_fs_d,
    X_val_fs_d,
    y_val_fs_d
)

print("\nSelected features from forward selection:")
print(forward_features_d)

Step 1: Added GenHlth, Validation RMSE = 0.6659091105159772
Step 2: Added HighBP, Validation RMSE = 0.6529849343290615
Step 3: Added BMI, Validation RMSE = 0.6468434206554213
Step 4: Added Age, Validation RMSE = 0.6425422447537571
Step 5: Added HighChol, Validation RMSE = 0.6398284726705386
Step 6: Added HeartDiseaseorAttack, Validation RMSE = 0.6381237637343247
Step 7: Added DiffWalk, Validation RMSE = 0.6370221292752614
Step 8: Added Income, Validation RMSE = 0.636434436730759
Step 9: Added HvyAlcoholConsump, Validation RMSE = 0.6359053459005342
Step 10: Added Sex, Validation RMSE = 0.6355783319361347
Step 11: Added CholCheck, Validation RMSE = 0.6352568989738724
Step 12: Added Stroke, Validation RMSE = 0.6350355077972498
Step 13: Added AnyHealthcare, Validation RMSE = 0.6350005312696907
Step 14: Added MentHlth, Validation RMSE = 0.6349663458077607
Step 15: Added PhysActivity, Validation RMSE = 0.634941522427062
Step 16: Added Veggies, Validation RMSE = 0.634934539402868
Step 17: Add

In [ ]:
# Evaluate forward selection model on the test set

forward_lr_d = LinearRegression()
forward_lr_d.fit(X_train_d[forward_features_d], y_train_d)

y_pred_forward_d = forward_lr_d.predict(X_test_d[forward_features_d])

forward_r2_d = r2_score(y_test_d, y_pred_forward_d)
forward_rmse_d = np.sqrt(mean_squared_error(y_test_d, y_pred_forward_d))

print("Diabetes Forward Selection R²:", forward_r2_d)
print("Diabetes Forward Selection RMSE:", forward_rmse_d)
print("Number of selected features:", len(forward_features_d))
print("Selected features:")
print(forward_features_d)

Diabetes Forward Selection R²: 0.173079467562387
Diabetes Forward Selection RMSE: 0.6323488738053289
Number of selected features: 17
Selected features:
['GenHlth', 'HighBP', 'BMI', 'Age', 'HighChol', 'HeartDiseaseorAttack', 'DiffWalk', 'Income', 'HvyAlcoholConsump', 'Sex', 'CholCheck', 'Stroke', 'AnyHealthcare', 'MentHlth', 'PhysActivity', 'Veggies', 'PhysHlth']


### Forward Selection Interpretation

For the diabetes dataset, forward selection chose 17 out of the 21 available predictors. The first selected feature was `GenHlth`, suggesting that general health was the strongest single predictor of diabetes status in this dataset. Other early selected features included `HighBP`, `BMI`, `Age`, and `HighChol`, which are all clinically reasonable predictors.

However, the forward selection model did not outperform the baseline linear regression model on the test set. Its R² was slightly lower and its RMSE was slightly higher. This suggests that removing a few predictors did not meaningfully improve predictive performance for this dataset, although the model became slightly simpler.

## Backward Selection: Diabetes Dataset

Backward selection starts with all available predictors and removes one feature at a time. At each step, the feature whose removal produces the lowest validation RMSE is removed. The process stops when removing another feature no longer improves validation performance.

In [18]:
# Backward selection function using validation RMSE

def backward_selection(X_train, y_train, X_val, y_val):
    selected_features = list(X_train.columns)
    best_rmse = float("inf")
    history = []

    # Initial model with all features
    initial_model = LinearRegression()
    initial_model.fit(X_train[selected_features], y_train)
    initial_pred = initial_model.predict(X_val[selected_features])
    best_rmse = np.sqrt(mean_squared_error(y_val, initial_pred))

    print(f"Initial model with all features, Validation RMSE = {best_rmse}")

    while len(selected_features) > 1:
        feature_results = []

        for feature in selected_features:
            current_features = [f for f in selected_features if f != feature]

            model = LinearRegression()
            model.fit(X_train[current_features], y_train)

            val_pred = model.predict(X_val[current_features])
            val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

            feature_results.append((feature, val_rmse))

        # Choose the feature whose removal gives the lowest validation RMSE
        feature_to_remove, current_best_rmse = min(feature_results, key=lambda x: x[1])

        # Stop if removing a feature does not improve validation RMSE
        if current_best_rmse < best_rmse:
            selected_features.remove(feature_to_remove)
            best_rmse = current_best_rmse

            history.append({
                "Step": len(history) + 1,
                "Removed Feature": feature_to_remove,
                "Validation RMSE": best_rmse,
                "Remaining Features": selected_features.copy()
            })

            print(f"Step {len(history)}: Removed {feature_to_remove}, Validation RMSE = {best_rmse}")
        else:
            break

    return selected_features, pd.DataFrame(history)


backward_features_d, backward_history_d = backward_selection(
    X_train_fs_d,
    y_train_fs_d,
    X_val_fs_d,
    y_val_fs_d
)

print("\nSelected features from backward selection:")
print(backward_features_d)

Initial model with all features, Validation RMSE = 0.63502895179662
Step 1: Removed Education, Validation RMSE = 0.6349843442815594
Step 2: Removed Fruits, Validation RMSE = 0.6349562670985783
Step 3: Removed NoDocbcCost, Validation RMSE = 0.6349412816290612
Step 4: Removed Smoker, Validation RMSE = 0.6349325137630388

Selected features from backward selection:
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Income']


In [19]:
# Evaluate backward selection model on the test set

backward_lr_d = LinearRegression()
backward_lr_d.fit(X_train_d[backward_features_d], y_train_d)

y_pred_backward_d = backward_lr_d.predict(X_test_d[backward_features_d])

backward_r2_d = r2_score(y_test_d, y_pred_backward_d)
backward_rmse_d = np.sqrt(mean_squared_error(y_test_d, y_pred_backward_d))

print("Diabetes Backward Selection R²:", backward_r2_d)
print("Diabetes Backward Selection RMSE:", backward_rmse_d)
print("Number of selected features:", len(backward_features_d))
print("Selected features:")
print(backward_features_d)

Diabetes Backward Selection R²: 0.173079467562387
Diabetes Backward Selection RMSE: 0.6323488738053289
Number of selected features: 17
Selected features:
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Income']


### Backward Selection Interpretation

For the diabetes dataset, backward selection removed 4 predictors: `Education`, `Fruits`, `NoDocbcCost`, and `Smoker`. The remaining 17 predictors were the same features selected by forward selection, although the two methods reached that result in different ways.

The backward selection model had the same R² and RMSE as the forward selection model. Like forward selection, it did not improve performance compared with the baseline linear regression model using all predictors. This suggests that the removed variables were not adding much predictive value, but keeping all features still produced the slightly best test performance.

## Principal Component Regression (PCR): Diabetes Dataset

Principal Component Regression first applies PCA to transform the original predictors into principal components. Linear regression is then fit using those components instead of the original features.

Because PCA is affected by feature scale, the predictors are standardized before applying PCA. I test different numbers of principal components and choose the number that gives the lowest validation RMSE.

In [20]:
# PCR: choose the best number of components using validation RMSE

max_components_d = X_train_d.shape[1]
pcr_results_d = []

for n_components in range(1, max_components_d + 1):
    pcr_model = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components)),
        ("regression", LinearRegression())
    ])

    pcr_model.fit(X_train_fs_d, y_train_fs_d)
    val_pred = pcr_model.predict(X_val_fs_d)

    val_rmse = np.sqrt(mean_squared_error(y_val_fs_d, val_pred))
    val_r2 = r2_score(y_val_fs_d, val_pred)

    pcr_results_d.append({
        "Components": n_components,
        "Validation R²": val_r2,
        "Validation RMSE": val_rmse
    })

pcr_results_df_d = pd.DataFrame(pcr_results_d)
pcr_results_df_d

,Components,Validation R²,Validation RMSE
0,1,0.126839,0.652494
1,2,0.148983,0.644167
2,3,0.149694,0.643898
3,4,0.151421,0.643244
4,5,0.151612,0.643172
5,6,0.165788,0.637775
6,7,0.167541,0.637105
7,8,0.167851,0.636986
8,9,0.167853,0.636985
9,10,0.168453,0.636756


In [21]:
# Find the best number of components based on validation RMSE

best_pcr_components_d = pcr_results_df_d.loc[
    pcr_results_df_d["Validation RMSE"].idxmin(),
    "Components"
]

print("Best number of PCR components for diabetes:", best_pcr_components_d)

Best number of PCR components for diabetes: 21


In [22]:
# Evaluate the best PCR model on the test set

best_pcr_model_d = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=int(best_pcr_components_d))),
    ("regression", LinearRegression())
])

best_pcr_model_d.fit(X_train_d, y_train_d)

y_pred_pcr_d = best_pcr_model_d.predict(X_test_d)

pcr_r2_d = r2_score(y_test_d, y_pred_pcr_d)
pcr_rmse_d = np.sqrt(mean_squared_error(y_test_d, y_pred_pcr_d))

print("Diabetes PCR R²:", pcr_r2_d)
print("Diabetes PCR RMSE:", pcr_rmse_d)
print("Number of PCR components:", int(best_pcr_components_d))

Diabetes PCR R²: 0.17330978646059458
Diabetes PCR RMSE: 0.6322608048598488
Number of PCR components: 21


### PCR Interpretation

For the diabetes dataset, PCR performed best on the validation set when all 21 principal components were included. This means that reducing the number of components did not improve model performance.

The final PCR model had the same R² and RMSE as the baseline linear regression model. This is expected because using all principal components preserves the same information as the original predictors, only in a transformed coordinate system. PCR would only produce a different result if some components were removed.

In this case, PCR did not improve prediction, but it helped show that the lower-dimensional component models lost useful information.

## Partial Least Squares Regression (PLSR): Diabetes Dataset

Partial Least Squares Regression is similar to PCR because it creates a smaller set of components from the predictors. However, unlike PCR, PLSR chooses components based on their relationship with the target variable. This means PLSR may perform better when the most predictive directions are not the same as the directions with the highest variance.

In [23]:
# PLSR: choose the best number of components using validation RMSE

max_pls_components_d = X_train_d.shape[1]
pls_results_d = []

for n_components in range(1, max_pls_components_d + 1):
    pls_model = Pipeline([
        ("scaler", StandardScaler()),
        ("pls", PLSRegression(n_components=n_components))
    ])

    pls_model.fit(X_train_fs_d, y_train_fs_d)
    val_pred = pls_model.predict(X_val_fs_d).ravel()

    val_rmse = np.sqrt(mean_squared_error(y_val_fs_d, val_pred))
    val_r2 = r2_score(y_val_fs_d, val_pred)

    pls_results_d.append({
        "Components": n_components,
        "Validation R²": val_r2,
        "Validation RMSE": val_rmse
    })

pls_results_df_d = pd.DataFrame(pls_results_d)
pls_results_df_d

,Components,Validation R²,Validation RMSE
0,1,0.155621,0.641650
1,2,0.171242,0.635687
2,3,0.172367,0.635256
3,4,0.172892,0.635054
4,5,0.172931,0.635039
5,6,0.172958,0.635029
6,7,0.172958,0.635029
7,8,0.172957,0.635029
8,9,0.172957,0.635029
9,10,0.172957,0.635029


In [24]:
# Find the best number of PLSR components based on validation RMSE

best_pls_components_d = pls_results_df_d.loc[
    pls_results_df_d["Validation RMSE"].idxmin(),
    "Components"
]

print("Best number of PLSR components for diabetes:", best_pls_components_d)

Best number of PLSR components for diabetes: 7


In [25]:
# Evaluate the best PLSR model on the test set

best_pls_model_d = Pipeline([
    ("scaler", StandardScaler()),
    ("pls", PLSRegression(n_components=int(best_pls_components_d)))
])

best_pls_model_d.fit(X_train_d, y_train_d)

y_pred_pls_d = best_pls_model_d.predict(X_test_d).ravel()

pls_r2_d = r2_score(y_test_d, y_pred_pls_d)
pls_rmse_d = np.sqrt(mean_squared_error(y_test_d, y_pred_pls_d))

print("Diabetes PLSR R²:", pls_r2_d)
print("Diabetes PLSR RMSE:", pls_rmse_d)
print("Number of PLSR components:", int(best_pls_components_d))

Diabetes PLSR R²: 0.17331190055839063
Diabetes PLSR RMSE: 0.6322599964179398
Number of PLSR components: 7


### PLSR Interpretation

For the diabetes dataset, PLSR selected 7 components as the best validation choice. This is much fewer than the original 21 predictors, which shows that PLSR was able to reduce the feature space while keeping nearly the same predictive performance.

The final PLSR model had a slightly higher R² and slightly lower RMSE than the baseline linear regression model, although the improvement was extremely small. This suggests that PLSR may be useful for simplifying the model, but it did not meaningfully improve prediction accuracy for this dataset.

Compared with PCR, PLSR reached near-baseline performance with fewer components because it uses information from the target variable when creating components.

In [26]:
# Compare diabetes model results

diabetes_results = pd.DataFrame({
    "Model": [
        "Baseline Linear Regression",
        "Forward Selection",
        "Backward Selection",
        "PCR",
        "PLSR"
    ],
    "R²": [
        baseline_r2_d,
        forward_r2_d,
        backward_r2_d,
        pcr_r2_d,
        pls_r2_d
    ],
    "RMSE": [
        baseline_rmse_d,
        forward_rmse_d,
        backward_rmse_d,
        pcr_rmse_d,
        pls_rmse_d
    ],
    "Features/Components Used": [
        X_train_d.shape[1],
        len(forward_features_d),
        len(backward_features_d),
        int(best_pcr_components_d),
        int(best_pls_components_d)
    ]
})

diabetes_results

,Model,R²,RMSE,Features/Components Used
0,Baseline Linear Regression,0.173310,0.632261,21
1,Forward Selection,0.173079,0.632349,17
2,Backward Selection,0.173079,0.632349,17
3,PCR,0.173310,0.632261,21
4,PLSR,0.173312,0.632260,7


### Diabetes Model Comparison Summary

The diabetes results show that all models performed very similarly. The baseline linear regression model and PCR had the same R² and RMSE because PCR used all 21 components. Forward and backward selection both selected 17 features and slightly underperformed the baseline model. PLSR had the best performance by a very small margin while using only 7 components.

Overall, PLSR was the most efficient model for the diabetes dataset because it reduced the number of components while maintaining nearly identical predictive performance. However, the differences between the models were extremely small, so none of the methods produced a major improvement over the baseline linear regression model.

## Alzheimer's Dataset

The second dataset used in this notebook is the Alzheimer's disease dataset. This dataset includes demographic, lifestyle, medical history, clinical, and cognitive variables that may be useful for predicting Alzheimer's diagnosis or disease-related outcomes.

In [27]:
# Load the Alzheimer's dataset

alzheimers_df = pd.read_csv("alzheimers_disease_data.csv")

# Display the first few rows
alzheimers_df.head()

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


In [28]:
# Check the shape and basic information of the Alzheimer's dataset

print("Alzheimer's dataset shape:", alzheimers_df.shape)
print("\nColumn names:")
print(alzheimers_df.columns)

print("\nMissing values:")
print(alzheimers_df.isnull().sum())

Alzheimer's dataset shape: (2149, 35)

Column names:
Index(['PatientID', 'Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI',
       'Smoking', 'AlcoholConsumption', 'PhysicalActivity', 'DietQuality',
       'SleepQuality', 'FamilyHistoryAlzheimers', 'CardiovascularDisease',
       'Diabetes', 'Depression', 'HeadInjury', 'Hypertension', 'SystolicBP',
       'DiastolicBP', 'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL',
       'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment',
       'MemoryComplaints', 'BehavioralProblems', 'ADL', 'Confusion',
       'Disorientation', 'PersonalityChanges', 'DifficultyCompletingTasks',
       'Forgetfulness', 'Diagnosis', 'DoctorInCharge'],
      dtype='str')

Missing values:
PatientID                    0
Age                          0
Gender                       0
Ethnicity                    0
EducationLevel               0
BMI                          0
Smoking                      0
AlcoholConsumption           0
PhysicalActivit

## Alzheimer's Modeling Setup

The target variable is `Diagnosis`, which indicates whether a patient has an Alzheimer's diagnosis. The `PatientID` column is removed because it is only an identifier, and `DoctorInCharge` is removed because it is a text/confidential placeholder rather than a meaningful numeric predictor.

The remaining variables are used as predictors in the regression models.

In [29]:
# Define features and target for the Alzheimer's dataset

X_alzheimers = alzheimers_df.drop(["Diagnosis", "PatientID", "DoctorInCharge"], axis=1)
y_alzheimers = alzheimers_df["Diagnosis"]

# Train-test split
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_alzheimers,
    y_alzheimers,
    test_size=0.2,
    random_state=42
)

print("Training feature shape:", X_train_a.shape)
print("Testing feature shape:", X_test_a.shape)
print("Training target shape:", y_train_a.shape)
print("Testing target shape:", y_test_a.shape)

Training feature shape: (1719, 32)
Testing feature shape: (430, 32)
Training target shape: (1719,)
Testing target shape: (430,)


## Baseline Linear Regression: Alzheimer's Dataset

I first fit a standard linear regression model using all available Alzheimer's predictors. This gives a baseline R² and RMSE that the feature selection and component-based models can be compared against.

In [30]:
# Baseline linear regression using all Alzheimer's predictors

baseline_lr_a = LinearRegression()
baseline_lr_a.fit(X_train_a, y_train_a)

# Make predictions
y_pred_baseline_a = baseline_lr_a.predict(X_test_a)

# Evaluate model
baseline_r2_a = r2_score(y_test_a, y_pred_baseline_a)
baseline_rmse_a = np.sqrt(mean_squared_error(y_test_a, y_pred_baseline_a))

print("Alzheimer's Baseline Linear Regression R²:", baseline_r2_a)
print("Alzheimer's Baseline Linear Regression RMSE:", baseline_rmse_a)

Alzheimer's Baseline Linear Regression R²: 0.4082396547722267
Alzheimer's Baseline Linear Regression RMSE: 0.3682901247121755


## Forward Selection: Alzheimer's Dataset

Forward selection begins with no predictors and adds one feature at a time. At each step, the model tests the remaining predictors and adds the feature that produces the lowest validation RMSE. The process stops when adding another feature no longer improves validation performance.

In [31]:
# Create a smaller training/validation split for Alzheimer's feature selection

X_train_fs_a, X_val_fs_a, y_train_fs_a, y_val_fs_a = train_test_split(
    X_train_a,
    y_train_a,
    test_size=0.2,
    random_state=42
)

print("Forward selection training shape:", X_train_fs_a.shape)
print("Forward selection validation shape:", X_val_fs_a.shape)

Forward selection training shape: (1375, 32)
Forward selection validation shape: (344, 32)


In [32]:
# Apply forward selection to the Alzheimer's dataset

forward_features_a, forward_history_a = forward_selection(
    X_train_fs_a,
    y_train_fs_a,
    X_val_fs_a,
    y_val_fs_a
)

print("\nSelected features from forward selection:")
print(forward_features_a)

Step 1: Added FunctionalAssessment, Validation RMSE = 0.42203056666864197
Step 2: Added ADL, Validation RMSE = 0.38129151002707934
Step 3: Added MMSE, Validation RMSE = 0.3674535225282348
Step 4: Added MemoryComplaints, Validation RMSE = 0.3491317527270655
Step 5: Added BehavioralProblems, Validation RMSE = 0.338854832064215
Step 6: Added CholesterolTriglycerides, Validation RMSE = 0.3381121517100745
Step 7: Added Age, Validation RMSE = 0.33749043490955233
Step 8: Added SleepQuality, Validation RMSE = 0.3369397026530463
Step 9: Added EducationLevel, Validation RMSE = 0.33644120866577043
Step 10: Added CholesterolHDL, Validation RMSE = 0.33594064986906114
Step 11: Added DiastolicBP, Validation RMSE = 0.33549700281012723
Step 12: Added Confusion, Validation RMSE = 0.3352466693825981
Step 13: Added DietQuality, Validation RMSE = 0.335014140223985
Step 14: Added Hypertension, Validation RMSE = 0.33484196762680757
Step 15: Added Gender, Validation RMSE = 0.33472217004333493
Step 16: Added F

In [33]:
# Evaluate forward selection model on the Alzheimer's test set

forward_lr_a = LinearRegression()
forward_lr_a.fit(X_train_a[forward_features_a], y_train_a)

y_pred_forward_a = forward_lr_a.predict(X_test_a[forward_features_a])

forward_r2_a = r2_score(y_test_a, y_pred_forward_a)
forward_rmse_a = np.sqrt(mean_squared_error(y_test_a, y_pred_forward_a))

print("Alzheimer's Forward Selection R²:", forward_r2_a)
print("Alzheimer's Forward Selection RMSE:", forward_rmse_a)
print("Number of selected features:", len(forward_features_a))
print("Selected features:")
print(forward_features_a)

Alzheimer's Forward Selection R²: 0.4155855538905887
Alzheimer's Forward Selection RMSE: 0.36599707594846753
Number of selected features: 19
Selected features:
['FunctionalAssessment', 'ADL', 'MMSE', 'MemoryComplaints', 'BehavioralProblems', 'CholesterolTriglycerides', 'Age', 'SleepQuality', 'EducationLevel', 'CholesterolHDL', 'DiastolicBP', 'Confusion', 'DietQuality', 'Hypertension', 'Gender', 'Forgetfulness', 'FamilyHistoryAlzheimers', 'HeadInjury', 'Ethnicity']


### Forward Selection Interpretation

For the Alzheimer's dataset, forward selection chose 19 out of the 32 available predictors. The first selected features were `FunctionalAssessment`, `ADL`, `MMSE`, `MemoryComplaints`, and `BehavioralProblems`, which are strongly related to cognitive and functional symptoms.

Unlike the diabetes dataset, forward selection improved performance compared with the baseline linear regression model. The forward selection model had a higher R² and lower RMSE while using fewer predictors. This suggests that removing weaker predictors helped reduce noise and improved generalization on the test set.

## Backward Selection: Alzheimer's Dataset

Backward selection starts with all available predictors and removes one feature at a time. At each step, the model removes the feature whose removal produces the lowest validation RMSE. The process stops when removing another feature no longer improves validation performance.

In [34]:
# Apply backward selection to the Alzheimer's dataset

backward_features_a, backward_history_a = backward_selection(
    X_train_fs_a,
    y_train_fs_a,
    X_val_fs_a,
    y_val_fs_a
)

print("\nSelected features from backward selection:")
print(backward_features_a)

Initial model with all features, Validation RMSE = 0.3405746024065573
Step 1: Removed Smoking, Validation RMSE = 0.339031385420073
Step 2: Removed AlcoholConsumption, Validation RMSE = 0.3375557813021668
Step 3: Removed Diabetes, Validation RMSE = 0.3364468439713618
Step 4: Removed Depression, Validation RMSE = 0.33573567624817036
Step 5: Removed PhysicalActivity, Validation RMSE = 0.3353829580890791
Step 6: Removed Disorientation, Validation RMSE = 0.33511197445166635
Step 7: Removed PersonalityChanges, Validation RMSE = 0.33486308867671816
Step 8: Removed BMI, Validation RMSE = 0.3346830906894835
Step 9: Removed CholesterolLDL, Validation RMSE = 0.33454123547527825
Step 10: Removed DifficultyCompletingTasks, Validation RMSE = 0.3344996458826864
Step 11: Removed SystolicBP, Validation RMSE = 0.33446797598482264
Step 12: Removed CardiovascularDisease, Validation RMSE = 0.3344511115039121
Step 13: Removed CholesterolTotal, Validation RMSE = 0.3344403079332146

Selected features from bac

In [35]:
# Evaluate backward selection model on the Alzheimer's test set

backward_lr_a = LinearRegression()
backward_lr_a.fit(X_train_a[backward_features_a], y_train_a)

y_pred_backward_a = backward_lr_a.predict(X_test_a[backward_features_a])

backward_r2_a = r2_score(y_test_a, y_pred_backward_a)
backward_rmse_a = np.sqrt(mean_squared_error(y_test_a, y_pred_backward_a))

print("Alzheimer's Backward Selection R²:", backward_r2_a)
print("Alzheimer's Backward Selection RMSE:", backward_rmse_a)
print("Number of selected features:", len(backward_features_a))
print("Selected features:")
print(backward_features_a)

Alzheimer's Backward Selection R²: 0.41558555389058904
Alzheimer's Backward Selection RMSE: 0.3659970759484674
Number of selected features: 19
Selected features:
['Age', 'Gender', 'Ethnicity', 'EducationLevel', 'DietQuality', 'SleepQuality', 'FamilyHistoryAlzheimers', 'HeadInjury', 'Hypertension', 'DiastolicBP', 'CholesterolHDL', 'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment', 'MemoryComplaints', 'BehavioralProblems', 'ADL', 'Confusion', 'Forgetfulness']


### Backward Selection Interpretation

For the Alzheimer's dataset, backward selection removed 13 predictors and kept 19 predictors. The final selected feature set matched the forward selection model, although the features were reached in a different order.

The backward selection model had the same R² and RMSE as the forward selection model. Both feature selection methods improved performance compared with the baseline linear regression model. This suggests that some of the original predictors added noise rather than useful predictive information.

## Principal Component Regression (PCR): Alzheimer's Dataset

For the Alzheimer's dataset, PCR is used to transform the original predictors into principal components. The features are standardized first because PCA is sensitive to scale. I test different numbers of principal components and choose the number that gives the lowest validation RMSE.

In [36]:
# PCR: choose the best number of components using validation RMSE

max_components_a = X_train_a.shape[1]
pcr_results_a = []

for n_components in range(1, max_components_a + 1):
    pcr_model = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components)),
        ("regression", LinearRegression())
    ])

    pcr_model.fit(X_train_fs_a, y_train_fs_a)
    val_pred = pcr_model.predict(X_val_fs_a)

    val_rmse = np.sqrt(mean_squared_error(y_val_fs_a, val_pred))
    val_r2 = r2_score(y_val_fs_a, val_pred)

    pcr_results_a.append({
        "Components": n_components,
        "Validation R²": val_r2,
        "Validation RMSE": val_rmse
    })

pcr_results_df_a = pd.DataFrame(pcr_results_a)
pcr_results_df_a

,Components,Validation R²,Validation RMSE
0,1,-0.000624,0.468738
1,2,0.056460,0.455171
2,3,0.064787,0.453158
3,4,0.102346,0.443965
4,5,0.112108,0.441545
5,6,0.119983,0.439582
6,7,0.181720,0.423883
7,8,0.182025,0.423803
8,9,0.179716,0.424401
9,10,0.205480,0.417683


In [37]:
# Find the best number of PCR components based on validation RMSE

best_pcr_components_a = pcr_results_df_a.loc[
    pcr_results_df_a["Validation RMSE"].idxmin(),
    "Components"
]

print("Best number of PCR components for Alzheimer's:", best_pcr_components_a)

Best number of PCR components for Alzheimer's: 31


In [38]:
# Evaluate the best PCR model on the Alzheimer's test set

best_pcr_model_a = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=int(best_pcr_components_a))),
    ("regression", LinearRegression())
])

best_pcr_model_a.fit(X_train_a, y_train_a)

y_pred_pcr_a = best_pcr_model_a.predict(X_test_a)

pcr_r2_a = r2_score(y_test_a, y_pred_pcr_a)
pcr_rmse_a = np.sqrt(mean_squared_error(y_test_a, y_pred_pcr_a))

print("Alzheimer's PCR R²:", pcr_r2_a)
print("Alzheimer's PCR RMSE:", pcr_rmse_a)
print("Number of PCR components:", int(best_pcr_components_a))

Alzheimer's PCR R²: 0.39104146104165205
Alzheimer's PCR RMSE: 0.373603560675554
Number of PCR components: 31


### PCR Interpretation

For the Alzheimer's dataset, PCR performed best on the validation set when 31 principal components were used. This means PCR only removed one component from the original 32 predictors.

However, the final PCR model did not outperform the baseline linear regression model on the test set. Its R² was lower and its RMSE was higher than the baseline. This suggests that transforming the predictors into principal components did not improve generalization for this dataset.

This result also shows one limitation of PCR: PCA chooses components based on variance in the predictors, not based on how strongly those components predict the target variable.

## Partial Least Squares Regression (PLSR): Alzheimer's Dataset

PLSR creates components that are chosen based on their relationship with the target variable. This can be useful when the directions that best explain variation in the predictors are not the same directions that best predict the outcome.

For the Alzheimer's dataset, I test different numbers of PLSR components and choose the number with the lowest validation RMSE.

In [39]:
# PLSR: choose the best number of components using validation RMSE

max_pls_components_a = X_train_a.shape[1]
pls_results_a = []

for n_components in range(1, max_pls_components_a + 1):
    pls_model = Pipeline([
        ("scaler", StandardScaler()),
        ("pls", PLSRegression(n_components=n_components))
    ])

    pls_model.fit(X_train_fs_a, y_train_fs_a)
    val_pred = pls_model.predict(X_val_fs_a).ravel()

    val_rmse = np.sqrt(mean_squared_error(y_val_fs_a, val_pred))
    val_r2 = r2_score(y_val_fs_a, val_pred)

    pls_results_a.append({
        "Components": n_components,
        "Validation R²": val_r2,
        "Validation RMSE": val_rmse
    })

pls_results_df_a = pd.DataFrame(pls_results_a)
pls_results_df_a

,Components,Validation R²,Validation RMSE
0,1,0.479706,0.338002
1,2,0.471566,0.340636
2,3,0.471608,0.340622
3,4,0.471763,0.340572
4,5,0.471758,0.340574
5,6,0.471755,0.340575
6,7,0.471755,0.340575
7,8,0.471755,0.340575
8,9,0.471755,0.340575
9,10,0.471755,0.340575


In [40]:
# Find the best number of PLSR components based on validation RMSE

best_pls_components_a = pls_results_df_a.loc[
    pls_results_df_a["Validation RMSE"].idxmin(),
    "Components"
]

print("Best number of PLSR components for Alzheimer's:", best_pls_components_a)

Best number of PLSR components for Alzheimer's: 1


In [41]:
# Evaluate the best PLSR model on the Alzheimer's test set

best_pls_model_a = Pipeline([
    ("scaler", StandardScaler()),
    ("pls", PLSRegression(n_components=int(best_pls_components_a)))
])

best_pls_model_a.fit(X_train_a, y_train_a)

y_pred_pls_a = best_pls_model_a.predict(X_test_a).ravel()

pls_r2_a = r2_score(y_test_a, y_pred_pls_a)
pls_rmse_a = np.sqrt(mean_squared_error(y_test_a, y_pred_pls_a))

print("Alzheimer's PLSR R²:", pls_r2_a)
print("Alzheimer's PLSR RMSE:", pls_rmse_a)
print("Number of PLSR components:", int(best_pls_components_a))

Alzheimer's PLSR R²: 0.4065477831812744
Alzheimer's PLSR RMSE: 0.3688162286254985
Number of PLSR components: 1


### PLSR Interpretation

For the Alzheimer's dataset, PLSR performed best on the validation set using only 1 component. This shows that PLSR was able to compress the predictors into a very small component space while still preserving much of the predictive information.

However, the final PLSR model did not outperform the baseline linear regression model on the test set. Its R² was slightly lower and its RMSE was slightly higher. This suggests that although one PLSR component was useful, reducing the model to one component may have removed some information that helped prediction.

Compared with PCR, PLSR performed better with far fewer components because it uses the relationship between the predictors and the target variable when creating components.

In [42]:
# Compare Alzheimer's model results

alzheimers_results = pd.DataFrame({
    "Model": [
        "Baseline Linear Regression",
        "Forward Selection",
        "Backward Selection",
        "PCR",
        "PLSR"
    ],
    "R²": [
        baseline_r2_a,
        forward_r2_a,
        backward_r2_a,
        pcr_r2_a,
        pls_r2_a
    ],
    "RMSE": [
        baseline_rmse_a,
        forward_rmse_a,
        backward_rmse_a,
        pcr_rmse_a,
        pls_rmse_a
    ],
    "Features/Components Used": [
        X_train_a.shape[1],
        len(forward_features_a),
        len(backward_features_a),
        int(best_pcr_components_a),
        int(best_pls_components_a)
    ]
})

alzheimers_results

,Model,R²,RMSE,Features/Components Used
0,Baseline Linear Regression,0.408240,0.368290,32
1,Forward Selection,0.415586,0.365997,19
2,Backward Selection,0.415586,0.365997,19
3,PCR,0.391041,0.373604,31
4,PLSR,0.406548,0.368816,1


### Alzheimer's Model Comparison Summary

The Alzheimer's dataset showed more variation across models than the diabetes dataset. Forward selection and backward selection performed the best overall, with the highest R² and lowest RMSE. Both methods selected 19 out of the original 32 predictors, suggesting that removing weaker predictors helped improve test performance.

PCR performed the worst of the five models. Even though it used 31 components, it had lower R² and higher RMSE than the baseline model. This suggests that the directions with the most variance in the predictors were not necessarily the most useful directions for predicting Alzheimer's diagnosis.

PLSR used only 1 component and performed close to the baseline model, but it did not outperform it. This shows that PLSR was useful for simplifying the model, but the feature selection methods were better for prediction in this dataset.

In [43]:
# Combine diabetes and Alzheimer's results into one table

diabetes_results_labeled = diabetes_results.copy()
diabetes_results_labeled.insert(0, "Dataset", "Diabetes")

alzheimers_results_labeled = alzheimers_results.copy()
alzheimers_results_labeled.insert(0, "Dataset", "Alzheimer's")

combined_results = pd.concat(
    [diabetes_results_labeled, alzheimers_results_labeled],
    ignore_index=True
)

combined_results

,Dataset,Model,R²,RMSE,Features/Components Used
0,Diabetes,Baseline Linear Regression,0.173310,0.632261,21
1,Diabetes,Forward Selection,0.173079,0.632349,17
2,Diabetes,Backward Selection,0.173079,0.632349,17
3,Diabetes,PCR,0.173310,0.632261,21
4,Diabetes,PLSR,0.173312,0.632260,7
5,Alzheimer's,Baseline Linear Regression,0.408240,0.368290,32
6,Alzheimer's,Forward Selection,0.415586,0.365997,19
7,Alzheimer's,Backward Selection,0.415586,0.365997,19
8,Alzheimer's,PCR,0.391041,0.373604,31
9,Alzheimer's,PLSR,0.406548,0.368816,1


## Overall Week 3 Model Comparison

Across both datasets, the results show that feature selection and dimensionality reduction did not affect each dataset in the same way.

For the diabetes dataset, all models performed almost identically. The baseline linear regression model, PCR, and PLSR had nearly the same R² and RMSE values. Forward and backward selection removed 4 predictors but slightly underperformed the baseline model. This suggests that feature selection did not meaningfully improve diabetes prediction, although PLSR was able to use only 7 components while maintaining nearly the same performance.

For the Alzheimer's dataset, forward and backward selection were more useful. Both methods selected 19 out of 32 predictors and improved performance compared with the baseline linear regression model. This suggests that some predictors in the Alzheimer's dataset may have added noise rather than useful predictive information. PCR performed worse than the baseline, which shows that the high-variance principal components were not necessarily the most predictive components. PLSR simplified the model to 1 component and performed close to baseline, but it did not outperform the feature selection methods.

Overall, forward and backward selection were most helpful for the Alzheimer's dataset, while PLSR was the most efficient dimensionality reduction method for the diabetes dataset. PCR was less useful in this notebook because removing principal components generally reduced predictive performance.

In [44]:
# Identify the best model for each dataset based on lowest RMSE

best_models = combined_results.loc[
    combined_results.groupby("Dataset")["RMSE"].idxmin()
]

best_models

,Dataset,Model,R²,RMSE,Features/Components Used
7,Alzheimer's,Backward Selection,0.415586,0.365997,19
4,Diabetes,PLSR,0.173312,0.632260,7


## Final Conclusion

For the diabetes dataset, PLSR had the lowest RMSE and highest R², but the improvement over baseline linear regression was extremely small. The main benefit of PLSR was that it used only 7 components instead of all 21 original predictors while keeping nearly identical performance. Forward and backward selection removed 4 predictors, but they did not improve test performance.

For the Alzheimer's dataset, forward and backward selection performed the best. Both selected 19 predictors and improved test performance compared with the baseline model. This suggests that feature selection was more useful for the Alzheimer's dataset than for the diabetes dataset. PCR performed worse than the baseline, while PLSR simplified the model but did not improve accuracy.

Overall, Week 3 showed that the usefulness of feature selection and dimensionality reduction depends on the dataset. Forward and backward selection helped when some predictors added noise, while PLSR helped reduce the number of predictors without losing much predictive power.